In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# -----------------------------------------------------------------------------
# CONFIGURATION
# -----------------------------------------------------------------------------
DATA_PATH = Path("data/books_dataset.csv")
# Switch to Path("data/thrillers.csv") if you want the thriller-only corpus.
TITLE_COL = "original_title"
DESCRIPTION_COL = "description"
AUTHOR_COL = "author"
TOP_N = 5
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CACHE_DIR = Path(".cache/book_similarity")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Change this to test different prompts.
QUERY_TEXT = (
    "A dark psychological thriller about a missing person, hidden secrets, "
    "and a tense investigation that keeps revealing disturbing details."
)


# -----------------------------------------------------------------------------
# LOAD + CLEAN CORPUS
# -----------------------------------------------------------------------------
def load_corpus(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)

    if DESCRIPTION_COL not in df.columns:
        raise KeyError(f"Missing required column: {DESCRIPTION_COL}")
    if TITLE_COL not in df.columns:
        raise KeyError(f"Missing required column: {TITLE_COL}")

    df = df.copy()
    df[DESCRIPTION_COL] = df[DESCRIPTION_COL].fillna("").astype(str).str.strip()
    df[TITLE_COL] = df[TITLE_COL].fillna("Unknown Title").astype(str).str.strip()

    if AUTHOR_COL in df.columns:
        df[AUTHOR_COL] = df[AUTHOR_COL].fillna("").astype(str).str.strip()

    df = df[df[DESCRIPTION_COL] != ""].reset_index(drop=True)
    print(f"Loaded {len(df):,} books from {path}")
    return df


# -----------------------------------------------------------------------------
# EMBEDDING CACHE
# -----------------------------------------------------------------------------
def cache_key(path: Path, model_name: str) -> str:
    return f"{path.stem}__{model_name.replace('/', '__')}"


def embedding_cache_paths(path: Path, model_name: str) -> tuple[Path, Path]:
    key = cache_key(path, model_name)
    embeddings_path = CACHE_DIR / f"{key}.npy"
    meta_path = CACHE_DIR / f"{key}.json"
    return embeddings_path, meta_path


def build_embeddings(df: pd.DataFrame, model_name: str, path: Path) -> np.ndarray:
    embeddings_path, meta_path = embedding_cache_paths(path, model_name)

    if embeddings_path.exists() and meta_path.exists():
        print(f"Using cached embeddings from {embeddings_path}")
        return np.load(embeddings_path)

    print(f"Loading transformer model: {model_name}")
    model = SentenceTransformer(model_name)
    texts = df[DESCRIPTION_COL].tolist()

    print(f"Encoding {len(texts):,} descriptions...")
    embeddings = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    np.save(embeddings_path, embeddings)
    meta_path.write_text(
        json.dumps(
            {
                "model_name": model_name,
                "source_path": str(path),
                "rows": len(df),
                "text_column": DESCRIPTION_COL,
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    print(f"Saved embeddings to {embeddings_path}")
    return embeddings


# -----------------------------------------------------------------------------
# COSINE SIMILARITY SEARCH
# -----------------------------------------------------------------------------
def search_books(
    query_text: str,
    df: pd.DataFrame,
    embeddings: np.ndarray,
    model_name: str,
    top_n: int = 5,
) -> pd.DataFrame:
    if not query_text.strip():
        raise ValueError("query_text cannot be empty")

    model = SentenceTransformer(model_name)
    query_embedding = model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0]

    scores = embeddings @ query_embedding
    top_indices = np.argsort(scores)[::-1][:top_n]

    results = df.iloc[top_indices].copy()
    results["similarity"] = scores[top_indices]
    results["rank"] = range(1, len(results) + 1)

    columns = ["rank", TITLE_COL, "similarity"]
    if AUTHOR_COL in results.columns:
        columns.insert(2, AUTHOR_COL)
    if DESCRIPTION_COL in results.columns:
        columns.append(DESCRIPTION_COL)

    return results[columns]


# -----------------------------------------------------------------------------
# RUN DEMO
# -----------------------------------------------------------------------------
df = load_corpus(DATA_PATH)
embeddings = build_embeddings(df, MODEL_NAME, DATA_PATH)
results = search_books(QUERY_TEXT, df, embeddings, MODEL_NAME, top_n=TOP_N)

print("\nQuery:")
print(QUERY_TEXT)
print("\nTop matches:")
print(results.to_string(index=False))


c:\Users\Admin\anaconda3\envs\capstone\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 13,337 books from data\books_dataset.csv
Using cached embeddings from .cache\book_similarity\books_dataset__sentence-transformers__all-MiniLM-L6-v2.npy


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3104.21it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Query:
A dark psychological thriller about a missing person, hidden secrets, and a tense investigation that keeps revealing disturbing details.

Top matches:
 rank  original_title          author  similarity                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

In [ ]:
QUERY_TEXT = (
    "A butler's love story set in 20th century England."
)
results = search_books(QUERY_TEXT, df, embeddings, MODEL_NAME, top_n=TOP_N)

print("\nQuery:")
print(QUERY_TEXT)
print("\nTop matches:")
print(results.to_string(index=False))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2711.66it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Query:
A dark psychological thriller about a missing person, hidden secrets, and a tense investigation that keeps revealing disturbing details.

Top matches:
 rank  original_title          author  similarity                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        